# Ricci Finance v12 — Lecture Notebook

## Dynamic Ricci-capital networks, Gaussian HMM regimes, and market stories

This notebook is the teaching companion to `app_v12.py`. It develops the model step by step instead of hiding the pipeline behind the Streamlit interface.

### Learning objectives

By the end of the notebook, you should be able to:

1. convert market prices into a rolling financial network;
2. understand correlation distance and capital-adjusted graph distance;
3. compute and interpret Ollivier–Ricci curvature;
4. distinguish observed geometry from Ricci-flow regularization;
5. build a graph-level time series from rolling frames;
6. fit a Gaussian Hidden Markov Model and inspect posterior probabilities;
7. explain how frame-to-frame statistics create an evidence-based market story;
8. visualize the full sequence as a stable 3-D animated manifold.

> The project is for research and education. It does not guarantee future returns.

## 1. System architecture

```mermaid
flowchart TD
    A[Prices and volume] --> B[Returns and dollar volume]
    B --> C[Rolling windows]
    C --> D[Correlation and financial distance]
    D --> E[Threshold or kNN graph]
    B --> F[Capital mass and edge transport]
    E --> G[Capital-adjusted graph]
    F --> G
    G --> H[Ollivier–Ricci curvature]
    H --> I[Graph statistics by frame]
    I --> J[Gaussian HMM]
    J --> K[State probabilities]
    I --> L[Frame comparison]
    K --> L
    L --> M[Market story]
    H --> N[Stable 3-D manifold]
    M --> N
```

The key conceptual separation is:

- **Ricci curvature** describes spatial network geometry;
- **rolling frames** create temporal observations;
- **HMM** summarizes hidden structural regimes;
- **story generation** translates measured changes into descriptive language.

## 2. Environment and imports

**Purpose:** load the actual v12 package used by the Streamlit application.

**Expected output:** package imports succeed and the notebook reports the main scientific-library versions.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import plotly
import matplotlib

# Allow the notebook to run from either the repository root or lecture/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ricci_finance").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from ricci_finance import *

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("NetworkX:", nx.__version__)
print("Plotly:", plotly.__version__)
print("Matplotlib:", matplotlib.__version__)

## 3. Create demonstration market data

**Purpose:** use reproducible synthetic prices and volumes before relying on an external data service.

**Input:** ticker list, number of business days, random seed.

**Output:** close prices, volume, and dollar volume.

Synthetic data is useful for learning because every student receives the same result.

In [ ]:
TICKERS = ["NVDA", "AMD", "MU", "ANET", "AAOI", "IONQ", "QBTS", "RGTI"]

close, volume, dollar_volume = make_demo_market_data(
    TICKERS,
    n_days=320,
    seed=7,
)

print("Close shape:", close.shape)
print("Volume shape:", volume.shape)
display(close.tail())

### Optional: switch to real market data

The following cell is intentionally not executed automatically. It requires network access and data availability from Yahoo Finance.

In [ ]:
# Uncomment to use real data.
# close, volume, dollar_volume = download_market_data(
#     TICKERS,
#     period="1y",
#     interval="1d",
# )

## 4. Validate data and calculate log returns

For ticker \(i\), the log return is

\[
r_i(t)=\log\left(rac{P_i(t)}{P_i(t-1)}ight).
\]

Log returns are additive through time and are more convenient than raw prices for correlation analysis.

In [ ]:
warnings = validate_market_data(close, volume, minimum_rows=60)
print("Validation warnings:", warnings or "none")

returns = prices_to_returns(close)
dollar_volume = dollar_volume.reindex(
    index=returns.index,
    columns=returns.columns,
)

print("Return shape:", returns.shape)
display(returns.tail())

## 5. Inspect market features

The feature layer calculates volatility, momentum, drawdown, dollar volume, and relative volume. These variables provide economic context even when the graph itself is built primarily from return correlation.

In [ ]:
market_features = compute_market_features(
    close=close,
    volume=volume,
    window=60,
)

print("Available feature tables:", list(market_features))
display(market_features["volatility"].tail())

## 6. Correlation and financial distance

The correlation distance is

\[
d_{ij}^{\mathrm{corr}}=\sqrt{2(1-\rho_{ij})}.
\]

Interpretation:

- \(\rho_{ij}\approx 1\): short distance;
- \(\rho_{ij}\approx 0\): intermediate distance;
- \(\rho_{ij}\approx -1\): long distance.

In [ ]:
WINDOW_SIZE = 60
window_returns = returns.iloc[-WINDOW_SIZE:]

corr = window_returns.corr(min_periods=20)
dist = financial_distance_from_corr(corr)

display(corr.round(3))
display(dist.round(3))

## 7. Build the financial graph

**Method:** each node is a ticker. Edges are selected by the chosen graph rule.

Here we use `knn+bridges`:

- k-nearest-neighbor edges preserve local relationships;
- a small number of additional bridge edges preserve important cross-cluster connections;
- overlap counts and confidence values are stored on each edge.

In [ ]:
G0, corr_frame, dist_frame = build_graph_from_window(
    window_returns,
    graph_mode="knn+bridges",
    k_neighbors=3,
    min_corr=0.05,
    max_bridges=3,
    min_pair_obs=20,
)

print("Nodes:", G0.number_of_nodes())
print("Edges:", G0.number_of_edges())
display(summarize_edges(G0))

## 8. Add capital mass and edge transport

For each node, v12 uses median dollar volume in the current window as a capital-mass proxy.

The edge transport proxy is

\[
F_{ij}=\sqrt{M_iM_j}\max(\rho_{ij},0).
\]

The effective distance is

\[
d_{ij}^{\mathrm{eff}}
=\frac{d_{ij}^{\mathrm{corr}}}{1+\alpha_c s_{ij}},
\]

where \(s_{ij}\) measures similarity in capital scale.

This is a modeling proxy—not direct evidence that cash physically moved from one stock to another.

In [ ]:
capital_window = dollar_volume.reindex(
    index=window_returns.index,
    columns=window_returns.columns,
)

G_capital = attach_capital_attributes(
    G0,
    capital_window,
    capital_alpha=0.35,
    use_capital_weighting=True,
)

display(node_capital_table(G_capital))
display(capital_flow_table(G_capital).head(10))

## 9. Compute Ollivier–Ricci curvature

Ollivier–Ricci curvature compares how probability mass around neighboring nodes is transported.

Practical interpretation in this project:

- **negative curvature:** stretched or bridge-like relationship;
- **near zero:** locally neutral geometry;
- **positive curvature:** cohesive or redundant local relationship.

The implementation uses `GraphRicciCurvature` when available and a fallback approximation otherwise.

In [ ]:
G_ricci = compute_ricci_curvature(
    G_capital,
    alpha=0.5,
    method="OTD",
    proc=1,
)

edge_table = summarize_edges(G_ricci)
display(edge_table.sort_values("ricciCurvature"))

## 10. View one 3-D Ricci-capital manifold

Visual encoding:

- node size: capital mass;
- node color: connected component;
- edge width: capital-flow proxy;
- edge color: Ricci curvature;
- z-axis: Ricci stress or capital mass.

In [ ]:
positions_3d = compute_stable_layout_3d(
    G_ricci,
    seed=42,
)

fig_single = visualize_network_3d(
    G_ricci,
    positions_3d=positions_3d,
    title="Selected 60-day Ricci-capital manifold",
    node_cluster=compute_components(G_ricci),
    z_mode="ricci_stress",
)
fig_single.show()

## 11. Ricci flow: geometric regularization, not forecasting

Ricci flow updates graph edge lengths according to curvature. In v12 it is used as a geometric experiment:

\[
w_{ij}^{(t+1)}=w_{ij}^{(t)}\left(1-\eta\kappa_{ij}^{(t)}ight).
\]

It should be interpreted as **regularized geometry**, not the future market state.

In [ ]:
G_flowed, flow_history = run_ricci_flow(
    G_ricci,
    iterations=8,
    step_size=0.25,
    alpha=0.5,
    method="OTD",
    proc=1,
)

display(flow_history)
display(compare_before_after_flow(G_ricci, G_flowed).head(10))
plot_ricci_flow_history(flow_history).show()

## 12. Build rolling frames

A rolling frame repeats the entire graph construction over a moving time window.

Example with a 60-day window and 5-day step:

```text
Frame 1: days 1–60
Frame 2: days 6–65
Frame 3: days 11–70
```

Each frame stores the graph, correlations, distances, clusters, capital attributes, curvature, and graph-level statistics.

In [ ]:
frames, starts = build_rolling_frames(
    returns=returns,
    window_size=60,
    step=5,
    max_frames=30,
    graph_mode="knn+bridges",
    k_neighbors=3,
    min_corr=0.05,
    max_bridges=3,
    min_pair_obs=20,
    alpha=0.5,
    method="OTD",
    proc=1,
    dollar_volume=dollar_volume,
    use_capital_weighting=True,
    capital_alpha=0.35,
)

print("Frames:", len(frames))
print("First frame date:", frames[0].stats.end_date)
print("Last frame date:", frames[-1].stats.end_date)

## 13. Graph-level time series

The HMM does not consume every edge directly. Instead, each frame becomes one observation vector containing graph statistics such as curvature, density, cluster count, edge stability, and capital concentration.

In [ ]:
feature_df = rolling_feature_table(frames)
display(feature_df.head())
plot_rolling_features(feature_df).show()

## 14. Gaussian Hidden Markov Model

Let \(S_t\) be a hidden market-structure state and \(Y_t\) be the observed graph-feature vector. The HMM models:

\[
P(S_t\mid S_{t-1})
\]

and

\[
P(Y_t\mid S_t).
\]

v12 standardizes the features, fits a diagonal-covariance Gaussian HMM, predicts the most likely state, and calculates posterior probabilities.

In [ ]:
hmm_df, regime_labels = compute_hmm_regimes(
    frames=frames,
    returns=returns,
    starts=starts,
    window_size=60,
    n_components=3,
    forward_days=5,
    random_state=42,
)

print("Regime labels:", regime_labels)
display(hmm_df.tail())
plot_hmm_regimes(hmm_df).show()

## 15. Posterior probabilities

A hard state label hides uncertainty. Posterior probabilities answer a richer question:

> How much probability does the model assign to each hidden regime at this frame?

A frame labeled “transition” with 52% confidence is qualitatively different from one labeled “transition” with 96% confidence.

In [ ]:
probability_columns = [
    column
    for column in hmm_df.columns
    if column.startswith("state_probability_")
]

if probability_columns:
    display(
        hmm_df[
            ["date", "hmm_state", "regime_name", "state_probability"]
            + probability_columns
        ].tail(10)
    )
else:
    print("Posterior columns are unavailable; check hmmlearn installation and frame count.")

## 16. Generate a frame-by-frame market story

For every frame \(t\), v12 compares graph statistics with frame \(t-1\):

\[
\Delta X_t=X_t-X_{t-1}.
\]

The story is generated only from measured thresholds. It can say that curvature weakened or fragmentation increased, but it does not claim that these changes caused a future price move.

In [ ]:
stories = build_frame_stories(frames)
story_df = frame_story_table(stories)

display(
    story_df[
        [
            "frame",
            "date",
            "regime_name",
            "state_probability",
            "headline",
            "strongest_fragile_edge",
            "strongest_capital_edge",
        ]
    ].tail(10)
)

print("Latest narrative:\n")
print(stories[-1].narrative)

## 17. Animate the HMM story manifold

The animated figure keeps a stable layout so movement reflects changes in graph structure rather than an entirely new random layout.

Each frame synchronizes:

- graph geometry;
- HMM state;
- posterior confidence;
- graph-change narrative;
- most fragile edge;
- dominant capital edge.

In [ ]:
base_graph = build_base_graph_for_layout(
    frames,
    all_nodes=returns.columns,
)

stable_positions = compute_stable_layout_3d(
    base_graph,
    seed=42,
)

animation = build_3d_ricci_capital_animation(
    frames,
    stable_positions,
    z_mode="ricci_stress",
    stories=stories,
    frame_duration_ms=700,
)

animation.show()

## 18. Selected-frame analysis

The Streamlit **Selected frame** slider chooses one rolling window for detailed inspection. Here we reproduce that behavior in the notebook.

In [ ]:
SELECTED_FRAME = len(frames) - 1
selected = frames[SELECTED_FRAME]
selected_story = stories[SELECTED_FRAME]

print("Frame:", SELECTED_FRAME + 1)
print("Date:", selected_story.date)
print("Regime:", selected_story.regime_name)
print("Confidence:", selected_story.state_probability)
print("Headline:", selected_story.headline)
print("Narrative:", selected_story.narrative)

display(summarize_edges(selected.G))
display(surgery_risk_direction_table(selected.G).head(10))

## 19. Research interpretation and limitations

Before using the results, remember:

1. correlation is not causation;
2. edge capital flow is a proxy, not direct fund-flow measurement;
3. HMM state numbers and labels are sample-dependent;
4. Ricci curvature describes graph geometry, not expected return;
5. Ricci flow is regularization, not a temporal forecast;
6. automated stories depend on thresholds in `story.py`;
7. changing tickers, windows, and graph construction can change the regimes;
8. all claims should be evaluated with walk-forward validation.

## 20. Exercises

1. Change the rolling window from 60 to 20 days. How do regime stability and edge stability change?
2. Compare `threshold`, `knn`, and `knn+bridges` graph modes.
3. Set `capital_alpha=0`. Which curvature edges change most?
4. Increase the number of HMM states from 3 to 4. Are the new states stable and interpretable?
5. Identify frames where HMM confidence is low. What does the graph look like?
6. Compare the most fragile edge with the largest capital-flow edge. Are they usually the same?
7. Design a walk-forward test for whether stress-state probabilities contain predictive information.
8. Replace the synthetic data with real market data and document data-quality failures.

### Suggested next research modules

- directed lead–lag graph;
- persistent cluster matching;
- walk-forward HMM validation;
- probabilistic graph-feature forecasting with Toto or GluonTS;
- HTML or MP4 animation export.